# Forecasting Chicago transit ridership: analysis notebook

This notebook walks through the full analysis behind the [live dashboard](https://mehtaaarush.github.io/transit-ridership-forecasting/).
All logic lives in the `ridership` package, so the notebook stays short and the numbers match the pipeline exactly.

**Question.** How many people will ride the CTA each day over the next two weeks? Staffing, service levels and
crowding plans all start from a daily demand forecast.

**Outline**
1. Data and quality checks
2. Twenty-five years of ridership and the post-pandemic recovery
3. Weekly, seasonal, holiday and weather effects
4. Forecasting setup: direct multi-horizon with a ratio-to-level target
5. Rolling-origin backtest of four models
6. Error analysis and feature importance
7. The forward 14-day forecast

In [1]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ridership import analysis, config, data, features
from ridership.backtest import run_backtest
from ridership.calendar_features import holiday_table
from ridership.evaluate import summarize
from ridership.export import MODEL_NAMES
from ridership.figures import _style, BLUE, ORANGE, INK, INK2

_style()
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Data and quality checks

Two public sources, both fetched by `ridership.data` (no API keys):

* **CTA daily boardings**: City of Chicago Socrata API, one row per service day since 2001 (bus, rail, total, and the service schedule type).
* **Daily weather for the Loop**: Open-Meteo historical archive (mean temperature, feels-like minimum, precipitation, snowfall).

In [2]:
rides = data.load_ridership()
weather = data.load_weather()
daily = features.build_daily(rides, weather)

print(f"Ridership: {rides.index.min():%Y-%m-%d} to {rides.index.max():%Y-%m-%d}, {len(rides):,} days")
print(f"Weather:   {weather.index.min():%Y-%m-%d} to {weather.index.max():%Y-%m-%d}")
print("Missing ridership days:", int(rides['total'].isna().sum()))
print("Bus + rail == total on every day:", bool(((rides['bus'] + rides['rail']) == rides['total']).all()))
rides.tail()

Ridership: 2001-01-01 to 2026-08-31, 9,374 days
Weather:   2001-01-01 to 2026-09-14
Missing ridership days: 0
Bus + rail == total on every day: True


,day_type,bus,rail,total
date,,,,
2026-08-27,W,723453,537646,1261099
2026-08-28,W,701608,532305,1233913
2026-08-29,A,487851,396159,884010
2026-08-30,U,360055,270477,630532
2026-08-31,W,643844,432794,1076638


The CTA runs a Sunday schedule (`day_type == "U"`) on six holidays, using the federally observed date when the holiday falls on a weekend.
The engineered `hol_major` flag must agree with the agency's own schedule on every weekday since 2019. The test suite checks this too.

In [3]:
recent = rides.loc["2019-01-01":]
flags = holiday_table(recent.index.min(), recent.index.max())
weekday = recent.index.day_of_week < 5
cta = set(recent.index[weekday & (recent["day_type"] == "U")])
ours = set(flags.index[(flags["hol_major"] == 1) & (flags.index.day_of_week < 5)])
print(f"{len(cta)} weekday Sunday-service days; calendar agrees exactly: {cta == ours}")

45 weekday Sunday-service days; calendar agrees exactly: True


## 2. Twenty-five years of ridership

In [4]:
hist = analysis.weekly_history(daily)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hist.index, hist["bus"], color=BLUE, lw=1.5, label="Bus")
ax.plot(hist.index, hist["rail"], color=ORANGE, lw=1.5, label="Rail")
ax.set_title("Boardings per day, weekly average")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1e3:.0f}K"))
ax.legend(loc="lower left")
plt.show()

annual = analysis.annual_totals(daily)
(annual / 1e6).round(1).rename(columns=str.title).tail(8).T

date,2018,2019,2020,2021,2022,2023,2024,2025
Bus,242.2000,237.3000,121.4000,117.4000,140.0000,161.7000,181.7000,199.6000
Rail,225.9000,218.5000,76.0000,78.6000,103.5000,117.4000,127.5000,138.4000
Total,468.1000,455.7000,197.5000,196.0000,243.5000,279.1000,309.2000,338.0000


In [5]:
kpis = analysis.headline_kpis(daily)
print(f"Rides in the last 12 months: {kpis['rides_12m']/1e6:.1f}M ({kpis['rides_12m_yoy']:+.1%} YoY)")
print(f"Share of the same 12 months in 2018-19: {kpis['recovery_12m_vs_2019']:.1%}")
for k, v in kpis["recovery_by_day_type"].items():
    print(f"  {k:<15} {v:.1%}")
print(f"Bus share of rides: {kpis['bus_share_12m']:.1%} (vs {kpis['bus_share_2019']:.1%} pre-pandemic)")

Rides in the last 12 months: 343.9M (+4.2% YoY)
Share of the same 12 months in 2018-19: 74.9%
  weekday         71.3%
  saturday        88.9%
  sunday_holiday  92.8%
Bus share of rides: 58.8% (vs 51.9% pre-pandemic)


**Takeaway.** Weekend demand is back to roughly 90% of 2019, while weekday demand has settled near 70%. The commute is what changed.
The level is still rising about 4% a year, so a forecasting model cannot assume a stable mean. That drives the ratio-to-level framing in section 4.

## 3. What moves daily demand

Effects are measured against a *typical* day: the median of the same weekday one to three weeks either side, skipping holidays.
This removes the weekly cycle and the slow trend. (It looks ahead, so it is only used for description, never for forecasting.)

In [6]:
grid = analysis.seasonality_grid(daily)
grid.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
grid.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(grid.values / 1e3, cmap="Blues", aspect="auto")
ax.set_xticks(range(12), grid.columns); ax.set_yticks(range(7), grid.index); ax.grid(False)
fig.colorbar(im, ax=ax, label="Avg boardings (thousands)")
ax.set_title("Average boardings by weekday and month (last 24 months, regular days)")
plt.show()

In [7]:
effects = pd.DataFrame(analysis.holiday_effects(daily)).set_index("name")
effects["effect"] = effects["effect"].map("{:+.0%}".format)
effects

,kind,effect,days
name,,,
Christmas Day,major,-70%,4
Thanksgiving Day,major,-68%,4
New Year's Day,major,-54%,4
MLK Day,minor,-49%,4
Labor Day,major,-44%,4
Independence Day,major,-41%,5
Memorial Day,major,-40%,4
Day after Thanksgiving,eve,-39%,4
Christmas Eve,eve,-38%,4


In [8]:
w = analysis.weather_effects(daily)
pd.concat({k: pd.DataFrame(v).set_index("band") for k, v in w.items()}).assign(effect=lambda d: d["effect"].map("{:+.1%}".format))

effect  days
            band                      
temperature Below 10°F    -17.7%    15
            10–32°F        -1.4%   181
            32–50°F        +0.6%   398
            50–70°F        +0.7%   485
            70–85°F        -0.1%   329
            Above 85°F     -2.6%     9
snow        No snow        +0.3%  1270
            Up to 1 in     -2.0%   123
            1–3 in         -5.5%    21
rain        Dry            +1.2%   682
            Under 0.1 in   +0.7%   291
            0.1–0.5 in     -1.4%   200
            0.5+ in        -6.0%    97

**Takeaways.** Holidays dominate: Christmas and Thanksgiving remove about 70% of riders, and MLK Day about half. Recurring events push the other way:
the St. Patrick's parade adds about 40% and the Marathon about 35%. Weather matters less on average, but extreme cold (mean below 10°F) cuts ridership by about 18%.

## 4. Forecasting setup

* **Direct multi-horizon.** Each row pairs a forecast origin *T* (last published day) with a target day *T + h*, *h* = 1…14.
* **Ratio-to-level target.** `log(y[T+h] / level[T])`, where `level` is the mean of the 28 days ending at *T* (four of each weekday).
  The model learns weekly, holiday and weather shapes, while the drifting level is carried by the origin.
* **No leakage.** Origin features only use data up to *T* (a unit test tampers with future values to prove it). Target-day features are the calendar,
  which is known in advance, and observed weather, which a live system would take from a weather forecast.

In [9]:
rows = features.make_supervised(daily, "2026-08-03", "2026-08-03")
rows[["origin", "target_date", "h", "level", "dow_ratio_recent", "momentum", "hol_major", "temp_mean_f", "y"]].head(8)

,origin,target_date,h,level,dow_ratio_recent,momentum,hol_major,temp_mean_f,y
0,2026-08-03,2026-08-04,1,"1,009,070.1786",1.1107,1.0513,0,75.8000,"1,150,147.0000"
1,2026-08-03,2026-08-05,2,"1,009,070.1786",1.1394,1.0513,0,74.5000,"1,094,367.0000"
2,2026-08-03,2026-08-06,3,"1,009,070.1786",1.1018,1.0513,0,72.9000,"1,117,868.0000"
3,2026-08-03,2026-08-07,4,"1,009,070.1786",1.0625,1.0513,0,76.1000,"1,069,847.0000"
4,2026-08-03,2026-08-08,5,"1,009,070.1786",0.8706,1.0513,0,75.0000,"891,065.0000"
5,2026-08-03,2026-08-09,6,"1,009,070.1786",0.7180,1.0513,0,71.8000,"601,671.0000"
6,2026-08-03,2026-08-10,7,"1,009,070.1786",1.0125,1.0513,0,73.6000,"964,115.0000"
7,2026-08-03,2026-08-11,8,"1,009,070.1786",1.1107,1.0513,0,74.2000,"982,710.0000"


## 5. Rolling-origin backtest

The final 12 months (Sep 2025 – Aug 2026) are forecast out-of-sample. At the start of each month every model is refit on data available at that time,
then forecasts every origin in that month for horizons 1–14. 80% prediction intervals come from residual quantiles per horizon on a one-year inner hold-out.

In [10]:
preds = run_backtest(daily)
overall = summarize(preds).set_index("model")
overall.index = overall.index.map(MODEL_NAMES)
overall[["wape", "mape", "mae", "rmse", "bias", "coverage", "interval_width"]].sort_values("wape")

,wape,mape,mae,rmse,bias,coverage,interval_width
model,,,,,,,
Gradient boosting,0.0427,0.0468,"40,166.6894","52,077.7039",-0.0006,0.7722,0.1388
Ridge regression,0.0572,0.0627,"53,844.0451","69,993.6590",0.0031,0.7972,0.1882
4-week weekday mean,0.0823,0.1018,"77,431.5415","119,421.6186",-0.0006,0.7710,0.2412
Seasonal naive,0.0960,0.1183,"90,287.9586","149,183.6316",-0.0014,0.7734,0.2774


In [11]:
by_h = summarize(preds, by=["h"]).pivot(index="h", columns="model", values="wape")
fig, ax = plt.subplots(figsize=(9, 3.6))
for key, color in zip(["gbm", "ridge", "mean4", "naive"], [BLUE, ORANGE, "#1baf7a", "#eda100"]):
    ax.plot(by_h.index, by_h[key] * 100, color=color, lw=2, label=MODEL_NAMES[key])
ax.set_xlabel("Days ahead"); ax.set_ylabel("WAPE (%)"); ax.set_ylim(bottom=0)
ax.set_title("Error by forecast horizon"); ax.legend(ncol=2, loc="lower right")
plt.show()

Gradient boosting has **roughly half the error of the better baseline at every horizon** (4.2–4.3% WAPE against 7.8–8.6%). Its accuracy
barely changes from 1 to 14 days ahead, because the target-day calendar and weather explain most of the variation once the level is fixed. Ridge regression sits in between: it gets the calendar right
but cannot capture interactions such as *holiday × weekday* or *cold × snow* without hand-built terms.

## 6. Error analysis

In [12]:
by_type = summarize(preds, by=["day_type"]).pivot(index="day_type", columns="model", values="wape")
by_type.index = by_type.index.map({"W": "Weekday", "A": "Saturday", "U": "Sunday / holiday"})
by_type[["gbm", "ridge", "mean4", "naive"]].rename(columns=MODEL_NAMES)

model,Gradient boosting,Ridge regression,4-week weekday mean,Seasonal naive
day_type,,,,
Saturday,0.0581,0.0591,0.0826,0.1006
Sunday / holiday,0.0675,0.0808,0.1598,0.1776
Weekday,0.0373,0.0540,0.0726,0.0851


In [13]:
g7 = preds[(preds["model"] == "gbm") & (preds["h"] == 7)].copy()
g7["error"] = g7["pred"] / g7["y"] - 1
misses = g7.reindex(g7["error"].abs().sort_values(ascending=False).index).head(8)
misses = misses.merge(daily[["temp_mean_f", "feels_like_min_f", "snow_in"]], left_on="target_date", right_index=True)
misses[["target_date", "day_type", "y", "pred", "error", "temp_mean_f", "feels_like_min_f", "snow_in"]]

,target_date,day_type,y,pred,error,temp_mean_f,feels_like_min_f,snow_in
6488,2025-12-14,U,"433,031.0000","271,187.6144",-0.3737,2.8000,-13.7000,0.0000
8350,2026-01-23,W,"387,388.0000","503,085.4405",0.2987,-4.8000,-21.6000,0.0000
16918,2026-07-04,A,"645,748.0000","787,979.2525",0.2203,74.3000,74.0000,0.0000
8392,2026-01-26,W,"808,114.0000","633,724.3944",-0.2158,3.4000,-12.6000,0.0000
8378,2026-01-25,U,"380,517.0000","456,880.1984",0.2007,14.0000,-8.2000,1.4060
16904,2026-07-03,U,"750,977.0000","602,293.6811",-0.1980,78.7000,78.9000,0.0000
15084,2026-05-24,U,"693,884.0000","567,582.4032",-0.1820,62.8000,59.4000,0.0000
3128,2025-10-18,A,"891,691.0000","732,177.1266",-0.1789,66.1000,61.1000,0.0000


(`error` is forecast ÷ actual − 1, so a positive value means the forecast was too high.)

The largest misses fall into two groups:

* **Extreme cold, in both directions.** The forecast was 37% too low on a December Sunday with a feels-like low of −14°F, and 30% too high
  on a January Friday at −22°F. Days this cold are rare in the training window, and how much travel they suppress depends on closures that
  are announced at short notice.
* **The July 4th weekend of 2026.** The holiday fell on a Saturday and was observed on Friday, a pattern the training window had never seen
  (every earlier weekend holiday since 2022 fell on a Sunday). The forecast was 20% too low on the Friday and 22% too high on the Saturday:
  riders split their travel across the two days differently than the model expected.

An events and closures feed would be the most valuable next data source.

**Which features does the model rely on?** The importance below is permutation importance, measured out-of-sample: train on data before the backtest,
then shuffle each feature in the backtest year and record how much WAPE worsens.

In [14]:
from ridership.models import GradientBoostingForecaster
from ridership.evaluate import forecast_metrics

all_rows = features.make_supervised(daily, config.MODEL_START, daily["total"].last_valid_index())
train = all_rows[all_rows["target_date"] < config.BACKTEST_START]
test = all_rows[all_rows["origin"] >= config.BACKTEST_START]
gbm = GradientBoostingForecaster().fit(train)
base = forecast_metrics(test["y"], gbm.predict(test))["wape"]

rng = np.random.default_rng(0)
imp = {}
for col in features.MODEL_FEATURES:
    scores = []
    for _ in range(3):
        shuffled = test.copy()
        shuffled[col] = rng.permutation(shuffled[col].to_numpy())
        scores.append(forecast_metrics(test["y"], gbm.predict(shuffled))["wape"] - base)
    imp[col] = np.mean(scores)
imp = pd.Series(imp).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(imp.index, imp.values * 100, color=BLUE, height=0.6)
ax.set_xlabel("WAPE increase when shuffled (percentage points)")
ax.set_title(f"Permutation importance (backtest year, base WAPE {base:.2%})")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
plt.show()
(imp.sort_values(ascending=False) * 100).round(2).head(8).rename("WAPE increase (pp)").to_frame()

,WAPE increase (pp)
dow,11.8300
doy,4.0800
dow_ratio_recent,2.7100
hol_major,1.0300
momentum,0.9800
precip_in,0.5500
temp_mean_f,0.4000
feels_like_min_f,0.2900


Day of week dominates, as expected, followed by day of year (the seasonal cycle) and the recent weekday profile (`dow_ratio_recent`),
which tracks how the weekday/weekend balance has shifted since the pandemic. Major holidays and recent momentum come next, then rain and
temperature. Temperature and feels-like temperature are strongly correlated, so shuffling either one alone understates their joint effect.
Snow barely registers, because the backtest year had few snowy days. (The base WAPE here is a little above the backtest's 4.3% because this
model is fit once, at the start of the year, rather than refit monthly.)

## 7. The forward forecast

The production model is fit on everything up to the last published day and forecasts the next 14 days. The dashboard lets you change the
weather or holiday status of any of those days and re-runs the exported trees in the browser.

In [15]:
from ridership.forecast import fit_final

model, quantiles, future = fit_final(daily)
out = future[["target_date", "dow", "hol_major", "temp_mean_f", "precip_in", "pred", "lo", "hi"]].copy()
out["dow"] = out["target_date"].dt.day_name().str[:3]
out[["pred", "lo", "hi"]] = out[["pred", "lo", "hi"]].round(-3).astype(int)
out

,target_date,dow,hol_major,temp_mean_f,precip_in,pred,lo,hi
0,2026-09-01,Tue,0,81.3000,0.0000,1200000,1117000,1294000
1,2026-09-02,Wed,0,83.9000,0.0080,1183000,1099000,1278000
2,2026-09-03,Thu,0,79.8000,0.4530,1190000,1108000,1288000
3,2026-09-04,Fri,0,78.7000,0.3230,1124000,1044000,1211000
4,2026-09-05,Sat,0,75.0000,0.0430,890000,829000,960000
5,2026-09-06,Sun,0,72.2000,0.0080,683000,637000,737000
6,2026-09-07,Mon,1,72.1000,0.0000,637000,595000,686000
7,2026-09-08,Tue,0,72.9000,0.0310,1244000,1161000,1333000
8,2026-09-09,Wed,0,73.7000,0.5710,1157000,1078000,1238000
9,2026-09-10,Thu,0,70.7000,0.0000,1224000,1140000,1311000


## Conclusions

* A ratio-to-level gradient-boosting model forecasts daily CTA ridership two weeks ahead with **~4.3% WAPE**. That is **56% less error** than a seasonal-naive
  forecast and **48% less** than the 4-week weekday average, validated on a full year of monthly refits.
* The model's errors are concentrated on rare, partly predictable shocks: extreme cold and unusual holiday configurations. Adding an events and closures
  calendar, plus weather *forecasts* rather than observed weather, are the clear next steps toward a live system.
* The weekday commute, not leisure travel, explains the gap to pre-pandemic ridership. That matters for where service is added back.